# Co-authorship Computational Social Science Network

## Outline

**Step 0**: Loading the data and build the network.

**Step 1**: Merging split author profiles.

**Step 2**: Filtering out peripheral nodes.

**Step 3**: Computing centrality metrics and assign visualization groups.

**Step 4**: Ranking authors by Betweenness per group.

## Step 0 — Load Data and Build Network

We load the global corpus and the CSS-classified subset, map CSS decisions, and construct the co-authorship graph.

In [ ]:
import json
import networkx as nx
import scipy.stats as stats
import time
from collections import defaultdict

# ─── FILE NAMES ───────────────────────────────────────────────────────────────
RAW_DATA_FILE = "global_classified.json"       # Raw file with 7000+ articles
CSS_DATA_FILE = "global_css_detected_v2.json"  # New file with 135 CSS articles
OUTPUT_GEXF   = "global_coauthorship_network_v2_2.gexf"

print("1. Loading data...")
with open(RAW_DATA_FILE, "r", encoding="utf-8") as f:
    global_corpus = json.load(f)

with open(CSS_DATA_FILE, "r", encoding="utf-8") as f:
    css_data = json.load(f)

css_map = {p["id"]: p.get("css_decision", "NO") for p in css_data}

# ─── 2. STATISTICAL ANALYSIS (SURFACE LEVEL) ──────────────────────────────────
paper_stats = []
for paper in global_corpus:
    paper_id = paper.get("id")
    css_decision = css_map.get(paper_id, "NO")
    is_css = (css_decision == "YES")
    
    authorships = paper.get("authorships", [])
    countries = set()
    for auth in authorships:
        for inst in auth.get("institutions", []):
            code = inst.get("country_code")
            if code: countries.add(code)
                
    has_tr = "TR" in countries
    has_international = len(countries - {"TR"}) > 0
    if has_tr:
        paper_stats.append({"paper_id": paper_id, "is_css": is_css, "has_intl": has_international})

# ─── 3. GRAPH CONSTRUCTION (FIRST PHASE: ID-BASED) ────────────────────────────
print("2. Building network topology...")
G = nx.Graph()

for paper in global_corpus:
    paper_id = paper.get("id")
    year = paper.get("publication_year", "")
    css = css_map.get(paper_id, "NO")
    authorships = paper.get("authorships", [])
    authors_in_paper = []

    for authorship in authorships:
        author = authorship.get("author", {})
        author_id = author.get("id", "")
        author_name = author.get("display_name", "Unknown")
        institutions = authorship.get("institutions", [])

        countries = list({inst.get("country_code", "") for inst in institutions if inst.get("country_code")})
        is_tr = "TR" in countries

        if author_id and author_id not in G:
            G.add_node(
                author_id,
                label=author_name,
                countries=",".join(sorted(countries)),
                is_tr=is_tr,
                has_css_paper=(css == "YES"),
                css_papers=[paper_id] if css == "YES" else [],
                total_papers=1,
            )
        elif author_id:
            G.nodes[author_id]["total_papers"] += 1
            if is_tr: G.nodes[author_id]["is_tr"] = True
            if countries:
                existing = set(G.nodes[author_id]["countries"].split(",")) if G.nodes[author_id]["countries"] else set()
                G.nodes[author_id]["countries"] = ",".join(sorted(list(existing | set(countries))))
            if css == "YES":
                G.nodes[author_id]["has_css_paper"] = True
                if paper_id not in G.nodes[author_id]["css_papers"]:
                    G.nodes[author_id]["css_papers"].append(paper_id)

        if author_id:
            authors_in_paper.append(author_id)

    # Edges
    for i in range(len(authors_in_paper)):
        for j in range(i + 1, len(authors_in_paper)):
            a, b = authors_in_paper[i], authors_in_paper[j]
            if G.has_edge(a, b):
                G[a][b]["weight"] += 1
                if paper_id not in G[a][b]["paper_ids"]: G[a][b]["paper_ids"].append(paper_id)
            else:
                G.add_edge(a, b, weight=1, paper_ids=[paper_id], css_edge=(css == "YES"), year=year if year else "")

## Step 1 — Author Profile Harmonization

We merge split OpenAlex profiles for the same author (e.g. one ID for TR affiliation, another for US).

In [ ]:
# ─── 🚀 NEW STEP: AUTHOR PROFILE HARMONIZATION ────────────────────────────────
print("3. Merging Turkey-linked split profiles (e.g. Onur Varol US/TR)...")
nodes_by_name = defaultdict(list)
for node, d in G.nodes(data=True):
    nodes_by_name[d["label"]].append((node, d))

merged_count = 0
for name, node_list in nodes_by_name.items():
    if len(node_list) > 1:
        if any(d["is_tr"] for n, d in node_list):
            master_node, master_data = node_list[0]
            for slave_node, slave_data in node_list[1:]:
                master_data["is_tr"] = master_data["is_tr"] or slave_data["is_tr"]
                master_data["has_css_paper"] = master_data["has_css_paper"] or slave_data["has_css_paper"]
                master_data["total_papers"] += slave_data["total_papers"]
                
                m_countries = set(master_data["countries"].split(",")) if master_data["countries"] else set()
                s_countries = set(slave_data["countries"].split(",")) if slave_data["countries"] else set()
                master_data["countries"] = ",".join(sorted(list((m_countries | s_countries) - {""})))
                
                master_data["css_papers"] = list(set(master_data["css_papers"] + slave_data["css_papers"]))
                
                neighbors = list(G.neighbors(slave_node))
                for neighbor in neighbors:
                    if neighbor == master_node: continue
                    edge_data = G[slave_node][neighbor]
                    if G.has_edge(master_node, neighbor):
                        G[master_node][neighbor]["weight"] += edge_data["weight"]
                        m_pids = G[master_node][neighbor]["paper_ids"]
                        G[master_node][neighbor]["paper_ids"] = list(set(m_pids + edge_data["paper_ids"]))
                    else:
                        G.add_edge(master_node, neighbor, **edge_data)
                
                G.remove_node(slave_node)
                merged_count += 1

print(f"   ✓ {merged_count} duplicate/split author profiles successfully merged!")

## Step 2 — Hairball Filter

We remove authors with fewer than 2 papers to reduce noise and isolate the active core.

In [ ]:
# ─── 🧹 NEW STEP: HAIRBALL FILTER (MINIMUM 2 PAPERS) ─────────────────────────
print("4. Cleaning up: Removing authors with only 1 paper from the network...")
single_paper_authors = [node for node, d in G.nodes(data=True) if d.get("total_papers", 0) < 2]
G.remove_nodes_from(single_paper_authors)
print(f"   ✓ Remaining active core author count: {G.number_of_nodes()}")

## Step 3 — Centrality Metrics and Visualization Groups

We compute Betweenness and Degree centrality, then assign each author to a viz_group based on CSS status and country affiliation.

In [ ]:
# ─── 5. CENTRALITY METRICS & VIZ_GROUP ASSIGNMENTS ────────────────────────────
print("5. Computing centrality metrics (Betweenness, Degree)...")
start_time = time.time()

for u, v, d in G.edges(data=True):
    d["distance"] = 1.0 / d["weight"]

degree = dict(G.degree())
betweenness = nx.betweenness_centrality(G, normalized=True, weight="distance")

for node, d in G.nodes(data=True):
    d["degree_centrality"] = degree[node]
    d["betweenness_centrality"] = round(betweenness[node], 6)
    
    # ─── NEW: viz_group and Color Assignment (Plasma Palette) ─────────────────
    if d["is_tr"] and d["has_css_paper"]:
        d["viz_group"] = 1   # CSS TR (Pinkish Core)
        d["color"] = "#cc4778"
    elif not d["is_tr"] and d["has_css_paper"]:
        d["viz_group"] = 2   # CSS Global-ex-TR (Yellowish Partners)
        d["color"] = "#f89540"
    else:
        d["viz_group"] = 3   # Non-CSS TR & Foreign (Purple Base)
        d["color"] = "#7e03a8" 

    d["css_papers"] = ",".join(d["css_papers"])

for u, v, d in G.edges(data=True):
    d["paper_ids"] = ",".join(d.get("paper_ids", [])) if isinstance(d.get("paper_ids"), list) else d.get("paper_ids", "")
    d["css_edge"] = int(d.get("css_edge", False))

print(f"   ✓ Computation complete! (Duration: {round((time.time() - start_time)/60, 2)} minutes)")

# ─── 6. SUMMARY & EXPORT ──────────────────────────────────────────────────────
print("\n" + " TOP 25 GLOBAL BETWEENNESS CENTRALITY (HARMONIZED) ".center(60, "-"))
top_bw = sorted(G.nodes(data=True), key=lambda x: -x[1]["betweenness_centrality"])[:25]
for node, d in top_bw:
    css_flag = "★CSS" if d["has_css_paper"] else "    "
    tr_flag = "TR" if d["is_tr"] else str(d["countries"])[:6]
    print(f"  {css_flag} bw={d['betweenness_centrality']:.6f}  [{tr_flag:5s}]  {d['label']} ({d['countries']})")

nx.write_gexf(G, OUTPUT_GEXF)
print(f"\nSUCCESS: {OUTPUT_GEXF} network backbone cleanly exported!")

## Step 4 — Betweenness Centrality Tables by Group

We sort authors within each viz_group by Betweenness Centrality and print ranked Markdown tables.

In [ ]:
# ─── BETWEENNESS CENTRALITY GROUPED TABLE GENERATOR ──────────────────────────

import pandas as pd

# 1. Split authors into buckets by viz_group
css_tr_authors = []       # viz_group = 1 (Pinkish)
css_global_authors = []   # viz_group = 2 (Yellowish)
non_css_authors = []      # viz_group = 3 (Purple)

for node, d in G.nodes(data=True):
    author_info = {
        "Name": d.get("label", "Unknown"),
        "Countries": d.get("countries", "TR" if d.get("is_tr") else "Global"),
        "Betweenness": d.get("betweenness_centrality", 0.0)
    }
    
    # Assign to the correct list by viz_group
    v_group = d.get("viz_group")
    if v_group == 1:
        css_tr_authors.append(author_info)
    elif v_group == 2:
        css_global_authors.append(author_info)
    else:
        non_css_authors.append(author_info)

# 2. Sort each group by Betweenness score descending
css_tr_sorted = sorted(css_tr_authors, key=lambda x: -x["Betweenness"])
css_global_sorted = sorted(css_global_authors, key=lambda x: -x["Betweenness"])
non_css_sorted = sorted(non_css_authors, key=lambda x: -x["Betweenness"])

# 3. Print clean Markdown/Overleaf-compatible tables
def generate_markdown_table(title, sorted_list, top_n=15):
    print(f"\n### {title}")
    print("| Rank | Author Name | Countries/Affiliation | Betweenness Centrality ($bw$) |")
    print("| :---: | :--- | :---: | :---: |")
    
    # Print placeholder if list is empty
    if not sorted_list:
        print("| - | *No authors matched this category* | - | - |")
        return
        
    for rank, author in enumerate(sorted_list[:top_n], 1):
        country_str = author['Countries'] if author['Countries'] else "N/A"
        print(f"| {rank} | **{author['Name']}** | {country_str} | {author['Betweenness']:.6f} |")

# 4. Run tables and print output
print("="*80)
print(" BETWEENNESS CENTRALITY ($Betweenness$) LISTS BY METHODOLOGICAL GROUP ".center(80, "═"))
print("="*80)

generate_markdown_table("GROUP 1: CSS TR Authors (Pinkish — #cc4778)", css_tr_sorted, top_n=10)
generate_markdown_table("GROUP 2: CSS Global-ex-TR Authors (Yellowish — #f89540)", css_global_sorted, top_n=10)
generate_markdown_table("GROUP 3: Non-CSS Authors — TR & Global Combined (Purple — #7e03a8)", non_css_sorted, top_n=25)